In [0]:
import os
import pandas as pd



import pyspark.sql.utils;
from pyspark.sql.types import StructType, StringType;
from pyspark.sql.functions import concat, lit, col
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType
from datetime import datetime, timedelta
from pyspark.sql.functions import regexp_replace
import numpy as np
from pyspark.sql import SparkSession
spark.conf.set("spark.sql.execution.arrow.enabled", "true")

In [0]:
try:
    verbose_mode = dbutils.widgets.get("verbose_mode");
except:
    verbose_mode = 'debug'
try:
    current_division = dbutils.widgets.get("division");
except:
    current_division = 'mal'
try:
    current_environment = dbutils.widgets.get("environment");
except:
    current_environment = 'dev'
try:
    current_project = dbutils.widgets.get("project");
except:
    current_project = 'maite_bi'
try:
    current_production_line = dbutils.widgets.get("production_line");
except:
    current_production_line = 'gains'

current_catalog = current_division + '_' + current_project + '_' + current_environment;

current_schema = current_production_line;

current_location = 'abfss://' + current_project + '@adlsdpcom'+ current_environment +f'data.dfs.core.windows.net/' + current_production_line + '/'

if verbose_mode == 'debug':
    display("Debug Mode")
    display(f"current_division : {current_division}")
    display(f"current_environment : {current_environment}")
    display(f"current_project : {current_project}")
    display(f"current_production_line : {current_production_line}")
    display(f"current_catalog : {current_catalog}")
    display(f"current_schema : {current_schema}")
    display(f"current_location : {current_location}")

In [0]:
def delta_table_sql_update_table(source_table, target_table, business_keys, excluded_fields, update_option, verbose_mode):
    """
    Met à jour la table cible avec les données de la table source en excluant les champs exclus et les champs absents de la cible.

    Args:
        source_table: référence de la delta table ou de la vue temporaire source
        target_table: référence de la delta cible
        business_keys: description de la clé de jointure sous la forme [('batch_id',)], ('...',), ('...',)]
        excluded_fields: description des champs à exclure, a minima, les champs d'horodatage [('created_at',), ('updated_at',), ('deleted_at',), ('max_date_time',), ...]
        update_option : mode de mise à jour souhaité ("update" : n'effectue que les mises à jour, "upsert" : effectue les mises à jour et insert, "merge" : effectue les mises à jour et insert et les delete)

    Returns:
    -1 : KO
    0  : No Data from source 
    1 : OK
    """
    function_result = -1
    schema = StructType().add("field_name", StringType())
    # Création des dataframes
    source_df = spark.sql(f"""SELECT * FROM {source_table} LIMIT 1""")
    if source_df.isEmpty():
        function_result = 0
        print("La source est vide")
    else:
        business_keys_df = spark.createDataFrame(business_keys, schema)
        excluded_fields_df = spark.createDataFrame(excluded_fields, schema)

        target_df = spark.sql(f"""SELECT * FROM {target_table} LIMIT 1""")
        target_schema = target_df.schema
        # Champs de la table cible
        target_fields_df = spark.createDataFrame([(c,) for c in target_df.columns], ["field_name"])
        # Typage de la source
        for column_name in source_df.columns:
            if column_name in target_fields_df.columns:
                source_df = source_df.withColumn(column_name, source_df[column_name].cast(target_schema[column_name].dataType)) 
        # Champs de la table source
        source_fields_df = spark.createDataFrame([(c,) for c in source_df.columns], ["field_name"])
        # Champs de la table cible
        target_fields_df = spark.createDataFrame([(c,) for c in target_df.columns], ["field_name"])
        # On supprime les champs absents dans la cible
        common_fields_df = source_fields_df.join(target_fields_df, "field_name", "left_semi")
        # On supprime les champs exclus
        all_fields_df = common_fields_df.join(excluded_fields_df, 'field_name', "left_anti")
        # Champs retenus hors clés
        field_list_df = all_fields_df.join(business_keys_df, 'field_name', "left_anti")
        # Création de la clause SELECT        
        select_clause_df = all_fields_df.withColumn('field_name', col('field_name'))
        select_clause_lst = [row.field_name for row in select_clause_df.collect()]
        select_clause_str = ", \n".join(select_clause_lst)
        # Création de la clause DE JOINTURE
        join_clause_df = business_keys_df.withColumn('field_name', concat(lit("target."), col('field_name'), lit(" = source."), col('field_name')))
        join_clause_lst = [row.field_name for row in join_clause_df.collect()]
        join_clause_str = "\n AND ".join(join_clause_lst)
        # Création de la liste des champs pour l'insertion
        insert_left_clause_df = all_fields_df.withColumn('field_name', col('field_name'))
        insert_left_clause_lst = [row.field_name for row in insert_left_clause_df.collect()]
        insert_left_clause_str = ", \n".join(insert_left_clause_lst)
        insert_left_clause_str += "\n, created_at"
        # Création de la liste des champs à insérer
        insert_right_clause_df = all_fields_df.withColumn('field_name', concat(lit("source."), col('field_name')))
        insert_right_clause_lst = [row.field_name for row in insert_right_clause_df.collect()]
        insert_right_clause_str = ", \n".join(insert_right_clause_lst)
        insert_right_clause_str += "\n, current_timestamp()"
        # Création de la condition pour l'insertion
        update_condition_clause_df = field_list_df.withColumn('field_name', concat(lit("(target."), col('field_name'), lit("<> source."), col('field_name'), lit(") or (target."), col('field_name'), lit(" IS NULL AND source."), col('field_name'), lit(" IS NOT NULL) OR (target."), col('field_name'), lit(" IS NOT NULL AND source."), col('field_name'), lit(" IS NULL)")))
        update_condition_clause_lst = [row.field_name for row in update_condition_clause_df.collect()]
        update_condition_clause_str = "\n OR ".join(update_condition_clause_lst)    #Problème ici
        # Création de la clause d'UPDATE
        update_clause_df = field_list_df.withColumn('field_name', concat(lit("target."), col('field_name'), lit(" = source."), col('field_name')))
        update_clause_lst = [row.field_name for row in update_clause_df.collect()]
        update_clause_str = ", \n ".join(update_clause_lst)
        update_clause_str += "\n, target.updated_at = current_timestamp()"
        #Définition de la requête de MERGE
        merge_query_sql = f"""MERGE INTO 
            {target_table} AS target
        USING
            {source_table} AS source
        ON {join_clause_str}
        WHEN MATCHED AND ({update_condition_clause_str}) 
            THEN UPDATE SET {update_clause_str}"""
        
        if update_option =="upsert" or update_option =="merge" :
            merge_query_sql = merge_query_sql + f"""
        WHEN NOT MATCHED BY TARGET 
            THEN INSERT ({insert_left_clause_str}) 
                VALUES ({insert_right_clause_str})"""
        
        if update_option =="merge" :
            merge_query_sql = merge_query_sql + f"""
        WHEN NOT MATCHED BY SOURCE AND deleted = FALSE
            THEN UPDATE SET
                deleted=TRUE,
                deleted_at = current_timestamp()
            """

        merge_query_sql = merge_query_sql + """
    ;"""

        if verbose_mode == 'debug':
            print('Requête de merge')
            display(merge_query_sql)  

        try:
            spark.sql(merge_query_sql)
            function_result = 1
        except Exception as e:
            print("erreur de merge:", e)
            function_result = -1
    return function_result

In [0]:
current_process="date_fin_production_temp"

In [0]:
configuration_file = f"""/Workspace/Shared/maite-bi/mal-ml-maite-bi/files/configuration/date_fin_production_temp.csv"""

if not os.path.exists(configuration_file):
    print(f"""Erreur : Le fichier {configuration_file} n'existe pas.""")
    dbutils.notebook.exit(-1)
else:
    print("Check file exists : OK")

In [0]:
configuration_file_df = pd.read_csv(configuration_file, sep=",")

configuration_file_df['done'] = pd.to_datetime(configuration_file_df['done'], errors='coerce')

if verbose_mode == 'debug':
    display(configuration_file)
    print( list(configuration_file_df.columns))
    print(configuration_file_df.dtypes)

In [0]:
spark.sql(f"TRUNCATE TABLE {current_catalog}.{current_schema}.{current_process}")

In [0]:
target_table_df = spark.sql(f"""SELECT * FROM {current_catalog}.{current_schema}.{current_process} WHERE 1=0""").toPandas()

if verbose_mode == 'debug':
    display(target_table_df.columns)
    print( list(target_table_df.columns))
    print(target_table_df.dtypes)

In [0]:
if not list(sorted(configuration_file_df.columns)) == list(sorted(target_table_df.columns)):
    print("Erreur : Le fichier n'a pas les mêmes colonnes que la table.")
    dbutils.notebook.exit(-1)
else:
    print("Check columns : OK")

In [0]:
configuration_file_sdf = spark.createDataFrame(configuration_file_df)

configuration_file_sdf.createOrReplaceTempView(f"v_ingestion_{current_process}")
if verbose_mode == 'debug' :
    display(configuration_file_sdf)

In [0]:
business_keys = [('batch_number',)]

In [0]:
excluded_fields = [('created_at',), ('updated_at',), ('deleted_at',)]

In [0]:
data_found = delta_table_sql_update_table(f"v_ingestion_{current_process}", f"{current_catalog}.{current_schema}.{current_process}", business_keys, excluded_fields, "upsert", verbose_mode)
if verbose_mode == 'debug':
    display(f"Merge Result : {data_found}") 
if data_found==-1:
    print("erreur de merge")
    dbutils.notebook.exit(-1)